In [1]:
import os
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec, Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
from pipeline.document_store import create_index
from pipeline.prompt import template
from pipeline.llm_call import llm
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from data.audio_text import transcribe_audio
from dotenv import load_dotenv

d:\notesRAG\notesRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
from langchain_classic.schema import Document

In [4]:
from data.audio_text import transcribe_audio

In [5]:
transcript = transcribe_audio(r"audio.mp3")

In [6]:
print(type(transcript))

<class 'str'>


In [7]:
print(transcript)

Speaker A: Jane?
Speaker B: Mark? Hi, it's been ages since I last saw you. How are you and Jackie?
Speaker A: Yeah, good, thanks.
Speaker B: And your new baby? George, isn't it?
Speaker A: Ha, you've got a good memory. Yes, he's 2 now. What about you? Are you still working in the health centre?
Speaker B: Yes, for the time being, but we're moving in a couple of months. Anyhow, I'd better go, I'm late for work. Lovely to see you again.
Speaker A: Yeah, likewise. Keep in touch.



In [8]:
doc = Document(page_content=transcript, metadata={"source": "audio_file_1"})

In [9]:
print(doc)

page_content='Speaker A: Jane?
Speaker B: Mark? Hi, it's been ages since I last saw you. How are you and Jackie?
Speaker A: Yeah, good, thanks.
Speaker B: And your new baby? George, isn't it?
Speaker A: Ha, you've got a good memory. Yes, he's 2 now. What about you? Are you still working in the health centre?
Speaker B: Yes, for the time being, but we're moving in a couple of months. Anyhow, I'd better go, I'm late for work. Lovely to see you again.
Speaker A: Yeah, likewise. Keep in touch.
' metadata={'source': 'audio_file_1'}


In [25]:
splitter = RecursiveCharacterTextSplitter(chunk_size=30, chunk_overlap=5)

docs = splitter.split_documents([doc])